# Visualising Terrascope data with TiTiler and JupyterGIS

In this notebook we combine two complementary tools to explore Terrascope data interactively from within JupyterLab: [JupyterGIS](https://jupytergis.readthedocs.io) for visualisation, and [TiTiler](https://developmentseed.org/titiler/) for serving the underlying imagery.

**JupyterGIS** is a JupyterLab extension that brings an interactive GIS environment directly into your notebook. It lets you visualise raster and vector layers on an interactive map, stack and reorder them, adjust their opacity from the layer panel, and load data from a wide range of formats such as **GeoJSON, GeoTIFF, Shapefile, GeoPackage, WMS/WMTS** and more. You can even add markers and story segments to tell interactive stories.

To feed JupyterGIS with Terrascope imagery, we rely on **TiTiler**, a modern map tile server that makes it easy to serve geospatial data on the web. Think of it as a specialised tool that takes large geographic files (like satellite imagery) and slices them into small, web-friendly map tiles that load efficiently in browser-based maps — exactly the kind of layers JupyterGIS can consume.

Throughout the notebook we use JupyterGIS as our interactive viewer to explore what the Terrascope TiTiler service has to offer, drawing the map through its **Python API**:

1. Introduction to the Terrascope TiTiler
2. Finding available datasets in TiTiler
3. Rendering WMTS layers from TiTiler in JupyterGIS
4. TiTiler COG endpoint
5. More JupyterGIS features


## 1. Introduction to the Terrascope TiTiler

[TiTiler](https://developmentseed.org/titiler/) is an open-source, FastAPI-based dynamic tile server for geospatial data. Instead of pre-generating tiles for every dataset, it reads Cloud Optimized GeoTIFFs (COGs) and STAC items on the fly and streams out map tiles, previews and statistics on demand.

The Terrascope TiTiler deployment at [titiler.terrascope.be](https://titiler.terrascope.be) exposes several standards-based endpoints on top of our data:

- **OGC WMS**: Serves georeferenced map images (e.g., PNG/JPEG) generated dynamically from spatial data.
- **OGC WMTS**: Serves pre-rendered, tiled map images for faster and more scalable map viewing.
- **COG endpoint**: direct access to any Cloud Optimized GeoTIFF by URL.

It is tightly integrated with the [Terrascope STAC API](https://stac.terrascope.be): TiTiler reads the asset file paths it needs for rendering, together with the product metadata (such as band information, nodata values and spatial extent), directly from STAC. This means you can point it at STAC collections and items and immediately get back web-ready tiles, without having to move or reprocess the underlying data.

The full interactive API specification is available at [titiler.terrascope.be/api.html](https://titiler.terrascope.be/api.html#/).

Lets define the base URL to the titiler service and its endpoints:


In [12]:
from yarl import URL
titiler_endpoint = URL("https://titiler.terrascope.be")
wms_endpoint = titiler_endpoint / "wms"
wmts_endpoint = titiler_endpoint / "wmts"
cog_endpoint = titiler_endpoint / "cog"

The health endpoint can be used to check if the TiTiler service is up and running, as well to check if the underlying STAC is working.

In [14]:
import requests
health_endpoint = titiler_endpoint / "healthz"
response = requests.get(str(health_endpoint))
if not response.ok:
    print(f"TiTiler service is not healthy: {response.status_code}")
else:
    print("Health check successful.")

Health check successful.


## 2. Discovering radar datasets on Terrascope

[Terrascope](https://terrascope.be) exposes its Earth Observation archive through a [STAC](https://stacspec.org) (SpatioTemporal Asset Catalog) API. Each **collection** describes the metadata of a dataset. As we said in the last section, the TiTiler service uses the Terrascope STAC collections to retrieve information for visualisations. Some of these collections support the [STAC Renders extension](https://github.com/stac-extensions/render), which contain the information needed by TiTiler to render the items of the collection.

If you want to explore the collections that you can have two options on how to find them:
1. Search the STAC API directly and finding the collections that support the Renders extension.
2. Use the Terrascope TiTiler GetCapabilities endpoint, which returns a list of all collections that support rendering as an XML.

Lets retrieve the renderable collections using GetCapabilities endpoint of the TiTiler WMTS service.

In [35]:
import xml.etree.ElementTree as ET

get_capabilities_endpoint = wmts_endpoint.with_query(service="WMTS", request="GetCapabilities")
print("Used endpoint:", get_capabilities_endpoint)
response = requests.get(str(get_capabilities_endpoint))
if response.ok:
    root = ET.fromstring(response.text)
    ns = {
        "wmts": "http://www.opengis.net/wmts/1.0",
        "ows": "http://www.opengis.net/ows/1.1",
    }
    layers = root.findall(".//wmts:Contents/wmts:Layer", ns)
    print(f"Found {len(layers)} renderable layers:\n")
    for layer in layers[:5]:
        identifier = layer.findtext("ows:Identifier", default="", namespaces=ns)
        title = layer.findtext("ows:Title", default="", namespaces=ns)
        print(f"- {title or identifier}")
    print("...")
else:
    print(f"Failed to retrieve capabilities: {response.status_code}")

Used endpoint: https://titiler.terrascope.be/wmts?service=WMTS&request=GetCapabilities
Found 117 renderable layers:

- aria-s1-gunw-v1_unfiltered_coherence
- cop-dem-glo-90m-cog_dem
- cropsar2d-fapar-archive_fapar
- cropsar2d-fapar-nrt_fapar
- esa-worldcereal-activecropland-10m-2021-v1_classification
...


The Terrascope TiTiler GetCapabilities endpoint can also be directly plugged into QGIS for discovering layers to render, more detailed information can be found here: https://docs.terrascope.be/Developers/WebServices/OGC/WMTSv2.html.

Jupytergis also supports the ´get_wms_available_layers´ which uses the GetCapabilities endpoint. However, due at the time of writing this function doesn't work with the Terrascope WMS.

## Rendering WMTS layers from TiTiler in JupyterGIS
Now that have found the available renderable layers, we can use JupyterGIS to visualise them. The following code snippet creates a new JupyterGIS document some of its layers to it.

First, lets create a new JupyterGIS document with a default view over Belgium. The `latitude`, `longitude` and `zoom` parameters control the initial camera position of the map. We will add openstreetmap as a basemap.

In [36]:
from jupytergis import GISDocument

doc = GISDocument(latitude=50.5, longitude=4.5, zoom=7)
doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap"
)
doc

JupyterGIS can consume any XYZ-style tile URL. The `{z}`, `{x}`, and `{y}` placeholders are filled in automatically by the map widget as the user pans and zooms, each tile request fetches exactly the pixels needed for the current view.

The helper function below constructs the correct Terrascope WMTS URL for a given render layer key. The optional `time` parameter lets us pin the tiles to a specific acquisition date, slicing into the dataset's time series.

In [46]:
def wmts_url_from_stac(layer: str, time: str = None) -> str:
    """Build a Terrascope WMTS XYZ URL from a render layer key and an optional date."""
    query = {
        "layer": layer,
        "style": "default",
        "tilematrixset": "EPSG:3857",
        "Service": "WMTS",
        "Request": "GetTile",
        "Version": "1.0.0",
        "Format": "image/png",
    }
    if time:
        query["TIME"] = time
    url = wmts_endpoint.with_query(query)
    return f"{url}&TileMatrix={{z}}&TileCol={{x}}&TileRow={{y}}"

'https://titiler.terrascope.be/wmts?layer=cop-dem-glo-90m-cog_dem&style=default&tilematrixset=EPSG:3857&Service=WMTS&Request=GetTile&Version=1.0.0&Format=image/png&TIME=2026-03-18&TileMatrix={z}&TileCol={x}&TileRow={y}'

Lets now render two random layers from the Terrascope TiTiler service. We will use the `wmts_url_from_stac` function to generate the correct WMTS URL for each layer, and then add them to the JupyterGIS document.

In [9]:
layer1 = ("cop-dem-glo-90m-cog_dem", "2012-11-20")
layer2 = ("aria-s1-gunw-v1_unfiltered_coherence", "2026-05-10")
doc.add_raster_layer(
    url=wmts_url_from_stac(layer1[0], time=layer1[1]),
    name="Layer 1"
)
doc.add_raster_layer(
    url=wmts_url_from_stac(layer2[0], time=layer2[1]),
    name="Layer 1",
    opacity=0.7
)
doc

'75729439-3288-4066-ac58-4df8872c50bb'

## TiTiler COG endpoint
Rendering WMTS layers is the most efficient way to visualise Terrascope data in JupyterGIS, but it is not always the most practical. The TiTiler service also exposes a COG endpoint that can be used to render any Cloud Optimized GeoTIFF directly, without having to go through WMTS. The COG endpoint can visualize any COG file that is mounted to the TiTiler. Lets start by cleaning up the previous document and creating a new one.

In [48]:
del doc
doc = GISDocument(latitude=50.5, longitude=4.5, zoom=7)
doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap"
)

'62dadb79-fefe-4d3f-88aa-b535905297c0'

## More JupyterGIS features

This notebook has only scratched the surface of what JupyterGIS can do. Here are some features worth exploring in the UI:

### 📂 Supported file formats
JupyterGIS can open and display many common geospatial formats:

| Format | Type | Notes |
|---|---|---|
| GeoJSON | Vector | Points, lines, polygons: load local files or remote URLs |
| GeoTIFF / COG | Raster | Local files or streamed from object storage |
| Shapefile, GeoPackage | Vector | Traditional GIS formats |
| WMS / WMTS | Raster service | OGC map and tile services (exactly what we used above) |
| XYZ tiles | Raster service | OpenStreetMap-style slippy map tiles |
| `.jGIS` | Project file | JupyterGIS's own format — saves all layers, styles, and camera position for later |

### 🎬 Story maps
**Story maps** let you turn a JupyterGIS map into a guided, step-by-step narrative, think of it as a presentation embedded in the map. Each story point captures:

- A specific **camera position** (centre coordinates + zoom level)
- Which **layers are visible** at that step
- A **text annotation** explaining what the viewer should focus on